# 42 · Sea-level heights at Manila's health facilities, against rasterio and SciPy

**Question.** Fieldwork experiment 198 built the sea-level example's third workflow by hand: three published grids
kept around Manila by Raster input, read at points by Sample raster, combined by the workflow's own rules, and set
against the AR6 rise at 2100 by the published ruleset *compare-level-to-elevation* 1.0.0. It found **16 of 194**
facilities on ground below that level. Does an independent reading of the same windows give the same heights and
the same 16?

**Independent.** rasterio reads the windows; SciPy's `RegularGridInterpolator` interpolates EGM2008 and DTU25MSS
between their published nodes; DeltaDTM is the cell rasterio's `index()` gives; the nearest site is a haversine
minimum in NumPy. Nothing of Fieldwork's code is read.

**Criteria, fixed before running.** At each AR6 site, EGM2008 and DTU25MSS within 1e-9 m of SciPy's value (none on
either side outside the window). At each facility, the ground exactly rasterio's cell and the same nearest site;
every height within 1e-9 m of the declared arithmetic; every level equal, and the count of Reaches.

In [ ]:
# Only what this notebook imports: rasterio cannot share this kernel with pyproj, fiona or geopandas (check 00).
NEEDED = ["numpy", "rasterio", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

## Parity

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_42_sea_level_heights import run

report = run(Path.cwd() / "fixtures" / "sea-level-heights-manila-001.json")
print("agrees:", report["agrees"])
print(f"Reaches: {report['reaches']} of {report['facilities']} here, {report['fieldworkReaches']} in Fieldwork")
m = report["manila"]
print(f"At Manila's site: EGM2008 {m['egm2008M']:.4f} m, DTU25MSS {m['mssM']:.4f} m, sea above the geoid {m['seaAboveGeoidM']:.4f} m")
for s in report["sites"]:
    print(("OK " if s["agrees"] else "XX ") + s["site_id"], "outside the window" if s["scipy"]["egm2008M"] is None else "")
print("differs:", report["differs"] or "nowhere")
print("within 1 mm of the projected level:", report["withinOneMillimetre"] or "none")

## What this cannot establish

**That a facility below the projected level will flood**: there is no tide, surge, defence or drainage here, and
DeltaDTM is least certain at the coast. **That the EGM2008 cut is PROJ's whole grid**: its value at the gauge,
43.009 m, is checked against pyproj in Fieldwork's own tests, since pyproj cannot share this kernel with rasterio.
**That OpenStreetMap's 194 facilities are all there are.**

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
e = report["environment"]
print("rasterio:", e["rasterio"], "· GDAL", e["gdal"], "· SciPy", e["scipy"], "· NumPy", e["numpy"])
print("runtime :", e["runtime"])